Here's everything organized in a logical learning order, grouped by topic so you can go through them one by one.

Group A — Core Spark Concepts
1. Small File Problem & coalesce()

The problem: When Spark writes output, it creates one file per partition. If you have many small partitions (e.g., from heavy filtering, or too many partitions set upstream), you end up with thousands of tiny files instead of a few well-sized ones.

Why it's bad:

Each file has overhead (metadata, open/close costs) — the NameNode (in HDFS) or object store gets overloaded tracking millions of small files.
Reading many small files later means many small tasks, which is slower than reading fewer large files.
Common cause: writing after a filter() that drastically reduces row count, but partition count stays the same.

How coalesce() helps:

python
df.coalesce(10).write.parquet("/output/path")

coalesce() reduces the number of output partitions without a full shuffle — it merges existing partitions together, so it's cheap. This means fewer, bigger output files.

Important distinction from repartition():

	coalesce(n)	repartition(n)
Shuffle?	No (or minimal)	Yes, full shuffle
Can increase partitions?	No — only decreases	Yes, can increase or decrease
Speed	Faster	Slower (more expensive)
Even distribution?	Not guaranteed	Guaranteed (evenly distributed)

Use coalesce() when reducing partitions before a write. Use repartition() when you need even distribution or need to increase partitions.

2. Tungsten Engine

What it is: Project Tungsten is Spark's memory and CPU optimization engine, introduced to make Spark run closer to raw hardware speed instead of relying on the JVM's default object handling.

What it does, specifically:

Off-heap memory management: Instead of storing data as regular Java objects (which have heavy overhead per object — headers, pointers, garbage collection costs), Tungsten stores data in a compact binary format directly in memory, closer to how C/C++ would lay out raw bytes.
Whole-stage code generation (codegen): Instead of interpreting each operation row by row through generic iterators, Tungsten generates actual Java bytecode at runtime for a whole stage of operations, collapsing multiple steps into a single tight loop — similar to what a hand-written optimized program would look like.
Cache-aware computation: Lays out data to take advantage of CPU cache lines (L1/L2 cache), reducing CPU cache misses.
Reduced garbage collection: Because data lives in compact binary form (not as millions of small Java objects), the JVM garbage collector has far less work to do.

Why it matters: Before Tungsten (Spark 1.x era), Spark was often slower than expected due to JVM/GC overhead. Tungsten is a core reason Spark SQL/DataFrame operations are significantly faster than old RDD-based code.

One-line interview answer: "Tungsten is Spark's internal engine that manages memory off-heap in binary format and generates optimized bytecode at runtime, to get closer to the performance of hand-written low-level code."

3. map() vs flatMap()

map(): Transforms each input element into exactly one output element. 1-to-1.

python
rdd = sc.parallelize(["hello world", "spark rocks"])
rdd.map(lambda x: x.split(" ")).collect()
# [['hello', 'world'], ['spark', 'rocks']]   -- one list per input line

flatMap(): Transforms each input element into zero, one, or many output elements, then flattens all results into a single flat collection. 1-to-many.

python
rdd.flatMap(lambda x: x.split(" ")).collect()
# ['hello', 'world', 'spark', 'rocks']   -- all words flattened into one list

Classic use case — Word Count (shown later in Q21) always uses flatMap, never map, because splitting a line into words naturally produces multiple outputs per input line that need to be flattened.

	map()	flatMap()
Output count per input	Exactly 1	0, 1, or many
Result structure	Same nesting as input	Flattened
Typical use	Transform each row	Split/explode each row into multiple rows
4. Driver OOM vs Executor OOM — Identifying via Spark UI

Driver OOM happens when too much data is pulled back to the driver — classic causes: collect() on a huge DataFrame, broadcasting a variable that's too large, or accumulating too much state in the driver.

How to spot it in Spark UI:

The application itself crashes entirely — not just one task, the whole job dies.
Driver logs show java.lang.OutOfMemoryError: Java heap space pointing to driver-side operations like collect().
In the Executors tab, the driver row shows abnormally high memory usage right before the crash.
Often no specific "failed task" — it's the driver process that dies, so the whole UI may become unreachable (since the UI is served by the driver).

Executor OOM happens when a task running inside an executor needs more memory than allocated — classic causes: large shuffles, skewed partitions (one partition way bigger than others), caching too much data, or a UDF with memory-heavy logic.

How to spot it in Spark UI:

Go to Jobs → Stages → that stage's task list. One or more specific tasks fail with ExecutorLostFailure or Container killed by YARN for exceeding memory limits.
Executors tab shows a specific executor marked as "dead," often with high GC time right before it died.
Stage detail page → look at the task duration/shuffle read size per task — if one task has a far larger shuffle read than others, that's data skew causing one task to OOM while others succeed.
Executor logs (stderr) show java.lang.OutOfMemoryError: GC overhead limit exceeded or Container killed by YARN.

Quick comparison:

	Driver OOM	Executor OOM
Scope of failure	Whole application dies	One or more tasks/executors die, job may recover
Common cause	collect(), broadcast too large	Data skew, large shuffle, caching too much
Where to look	Driver logs, UI becomes unreachable	Stage → task list, specific executor logs
Fix	Avoid collect(), increase spark.driver.memory, use take()/show() instead	Repartition to fix skew, increase spark.executor.memory, salting technique
5. spark-submit Command — Simple Example
bash
spark-submit \
  --master spark://master:7077 \
  --deploy-mode cluster \
  --executor-memory 4G \
  --executor-cores 2 \
  --num-executors 5 \
  --class com.example.MainApp \
  my_app.jar

For PySpark:

bash
spark-submit --master yarn --deploy-mode cluster my_script.py
Group B — Unix Commands (File Searching)
6. List All Files Containing a Specific Word
bash
grep -l "word" *

-l (lowercase L) prints only the filenames that contain a match, not the matching lines.

Recursively through subfolders:

bash
grep -rl "word" /path/to/folder
7. Print Files, Sub-files, and Directories Containing a Word
bash
grep -rl "word" /path/to/folder

-r = recursive (goes into subdirectories), -l = filenames only.

To also see the matching line content, not just filenames:

bash
grep -rn "word" /path/to/folder

Alternative using find + grep:

bash
find /path/to/folder -type f -exec grep -l "word" {} \;
Group C — SQL / PySpark Data Problems
8. Cumulative Sum (as of last date of every month)
python
from pyspark.sql import Window
from pyspark.sql.functions import sum, last_day, col

# Running total over the whole dataset, ordered by date
windowSpec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df = df.withColumn("cumulative_sum", sum("amount").over(windowSpec))

# Keep only the last date of each month
df_month_end = df.withColumn("month_end", last_day(col("date"))) \
                  .filter(col("date") == col("month_end"))

Equivalent SQL:

sql
SELECT *, SUM(amount) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_sum
FROM transactions
WHERE date = LAST_DAY(date)
9. Missing Monthly Transactions — How to Handle

The problem: If a month has zero transactions, it simply won't appear as a row at all — there's nothing to aggregate. You need to generate all expected months and left join your actual data onto that.

python
from pyspark.sql.functions import sequence, explode, date_format, to_date, lit, coalesce

# Generate a full calendar of months
calendar_df = spark.sql("""
    SELECT explode(sequence(to_date('2026-01-01'), to_date('2026-12-01'), interval 1 month)) AS month
""")

# Aggregate actual transactions by month
actuals_df = df.groupBy(date_format("date", "yyyy-MM-01").alias("month")).agg(sum("amount").alias("total"))

# Left join calendar onto actuals, fill missing months with 0
result = calendar_df.join(actuals_df, "month", "left") \
    .withColumn("total", coalesce(col("total"), lit(0)))

This is the standard pattern: never aggregate alone — always join against a complete reference calendar so missing periods show up as 0, not as absent rows.

10. CDC Mapping Using SQL

(Already covered in detail earlier, but the core merge logic in SQL:)

sql
MERGE INTO target_table t
USING cdc_source s
ON t.id = s.id
WHEN MATCHED AND s.operation = 'D' THEN DELETE
WHEN MATCHED AND s.operation = 'U' THEN UPDATE SET t.col1 = s.col1, t.col2 = s.col2
WHEN NOT MATCHED AND s.operation = 'I' THEN INSERT (id, col1, col2) VALUES (s.id, s.col1, s.col2)
11. Delete Records Older Than 30 Days
sql
DELETE FROM my_table
WHERE created_date < current_date() - INTERVAL 30 DAYS

In PySpark with Delta:

python
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, "/path/to/table")
dt.delete("created_date < date_sub(current_date(), 30)")
12. Double Delimiter Handling

The problem: A file using || or ,, as the delimiter instead of a single character — the default CSV reader can't split on multi-character delimiters directly.

Option 1 — read as text, split manually:

python
raw = spark.read.text("file.csv")
df = raw.withColumn("cols", split(col("value"), r"\|\|"))  # escape regex special chars
df = df.select([col("cols")[i].alias(f"col{i}") for i in range(num_cols)])

Option 2 — Spark's CSV reader actually supports multi-char delimiters natively (Spark 3+):

python
df = spark.read.option("delimiter", "||").option("header", True).csv("file.csv")
13. Deduplication — Keep Only the Latest Record
python
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

windowSpec = Window.partitionBy("id").orderBy(col("updated_at").desc())
df_dedup = df.withColumn("rn", row_number().over(windowSpec)) \
             .filter(col("rn") == 1) \
             .drop("rn")

SQL equivalent:

sql
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC) AS rn
    FROM my_table
) WHERE rn = 1
14. Three Consecutive Absences
python
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, datediff, sum as spark_sum

w = Window.partitionBy("student_id").orderBy("date")
df = df.filter(col("status") == "Absent")
df = df.withColumn("prev_date", lag("date", 1).over(w))
df = df.withColumn("gap", datediff(col("date"), col("prev_date")))
df = df.withColumn("new_group", (col("gap") != 1).cast("int"))
df = df.withColumn("group_id", spark_sum("new_group").over(w.rowsBetween(Window.unboundedPreceding, 0)))

result = df.groupBy("student_id", "group_id").count().filter(col("count") >= 3)

Logic: group consecutive absent dates together (gap resets the group), then filter groups with 3+ rows.

15. Remove Malformed/Bad Records
python
# While reading — quarantine bad records instead of crashing
df = spark.read.option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv("file.csv", header=True)

good_df = df.filter(col("_corrupt_record").isNull())
bad_df = df.filter(col("_corrupt_record").isNotNull())

Modes available: PERMISSIVE (default, nulls out bad fields, keeps row), DROPMALFORMED (drops bad rows entirely), FAILFAST (throws an error immediately on bad data).

16. Second Highest Salary
python
from pyspark.sql.functions import col, dense_rank
from pyspark.sql import Window

w = Window.orderBy(col("salary").desc())
df.withColumn("rnk", dense_rank().over(w)).filter(col("rnk") == 2)

SQL:

sql
SELECT DISTINCT salary FROM employees
ORDER BY salary DESC
LIMIT 1 OFFSET 1

dense_rank() is preferred over row_number() here — it correctly handles ties (two people with the same highest salary both count as rank 1, not 1 and 2).

17. Third Highest Salary

Same pattern, just change the filter:

python
df.withColumn("rnk", dense_rank().over(w)).filter(col("rnk") == 3)
sql
SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 1 OFFSET 2
18. Monthly Sales Report (Include Months With No Sales)

Same pattern as Q9 — generate the full month calendar, left join actual sales:

python
calendar_df = spark.sql("""
    SELECT date_format(explode(sequence(to_date('2026-01-01'), to_date('2026-12-01'), interval 1 month)), 'MMM') AS month
""")
sales_df = df.groupBy(date_format("sale_date", "MMM").alias("month")).agg(sum("amount").alias("total_sales"))

result = calendar_df.join(sales_df, "month", "left") \
    .withColumn("total_sales", coalesce(col("total_sales"), lit(0)))
19. Process a Specific File From a Directory
python
# Read just one named file
df = spark.read.csv("/data/folder/specific_file.csv", header=True)

# Read files matching a pattern
df = spark.read.csv("/data/folder/sales_2026*.csv", header=True)

# Read with a glob filter option
df = spark.read.option("pathGlobFilter", "*.csv").csv("/data/folder/")
20. Employee Earning More Than Their Manager
sql
SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS manager_salary
FROM employees e
JOIN employees m ON e.manager_id = m.id
WHERE e.salary > m.salary

This is a self-join — the employees table joined to itself, matching each employee's manager_id to the manager's own id.

PySpark equivalent:

python
emp = df.alias("e")
mgr = df.alias("m")
result = emp.join(mgr, col("e.manager_id") == col("m.id")) \
    .filter(col("e.salary") > col("m.salary")) \
    .select(col("e.name").alias("employee"), col("e.salary"), col("m.name").alias("manager"))
21. Word Count — PySpark and SQL

PySpark (RDD style, classic version):

python
rdd = sc.textFile("file.txt")
word_counts = rdd.flatMap(lambda line: line.split(" ")) \
                  .map(lambda word: (word, 1)) \
                  .reduceByKey(lambda a, b: a + b)
word_counts.collect()

PySpark (DataFrame style):

python
df = spark.read.text("file.txt")
words_df = df.withColumn("word", explode(split(col("value"), " ")))
word_counts = words_df.groupBy("word").count()

SQL:

sql
SELECT word, COUNT(*) AS count
FROM (
    SELECT explode(split(value, ' ')) AS word FROM text_table
)
GROUP BY word
22. Convert All 50 Columns to StringType
python
from pyspark.sql.functions import col

df = df.select([col(c).cast("string").alias(c) for c in df.columns])

This loops over every column name and casts each to StringType, regardless of original type — no need to list columns manually.

Group D — Advanced/Tricky PySpark Scenarios
23. Replace ID Columns With Their Description (Lookup Table With 100 Columns)

The scenario: table_1 has columns like dept, team, sub_team storing integer IDs, and a separate lookup table maps id → desc. With potentially many such columns (up to ~100), you don't want to write 100 manual joins.

python
from pyspark.sql.functions import col

# lookup_df has columns: id, desc
id_columns = ["dept", "team", "sub_team"]   # extend this list dynamically if needed

result_df = table_1
for c in id_columns:
    lookup_renamed = lookup_df.withColumnRenamed("id", f"{c}_id").withColumnRenamed("desc", f"{c}_desc")
    result_df = result_df.join(lookup_renamed, result_df[c] == lookup_renamed[f"{c}_id"], "left") \
        .drop(c, f"{c}_id") \
        .withColumnRenamed(f"{c}_desc", c)

Core idea: loop through each ID column, join against the same lookup table each time (aliased per column to avoid name collisions), replace the ID with the description, drop the helper columns. This scales to however many columns you have without hardcoding each join.

24. Write a UDF Function
python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def classify_salary(salary):
    if salary is None:
        return "Unknown"
    elif salary > 50000:
        return "High"
    else:
        return "Low"

classify_udf = udf(classify_salary, StringType())

df = df.withColumn("salary_band", classify_udf(col("salary")))

Always check first if a built-in function (when/otherwise) can do the job instead — UDFs are slower since they require JVM ↔ Python serialization per row (covered in detail earlier in this conversation).

25. Eliminate Duplicates WITHOUT Window Functions or dropDuplicates()

Using groupBy + aggregation instead:

python
from pyspark.sql.functions import max as spark_max, first

dedup_df = df.groupBy("id").agg(
    *[spark_max(c).alias(c) for c in df.columns if c != "id"]
)

Using groupBy to keep the actual full latest row (more correct version):

python
from pyspark.sql.functions import struct

dedup_df = df.groupBy("id").agg(
    spark_max(struct(col("updated_at"), *[col(c) for c in df.columns])).alias("latest")
).select("latest.*")

Why this trick works: struct() bundles all columns into one comparable value, orderBy-like comparison happens field-by-field starting with updated_at (the first field), so max() on the struct naturally picks the row with the latest updated_at — without ever using row_number() or dropDuplicates().

26. Join Two Tables With Inconsistent Zero-Padded IDs (0001 vs 1) — No Casting, No Substring

The constraint: can't use CAST(id AS INT) and can't use substring/lpad/ltrim to strip padding, since padding is inconsistent (some have 3 zeros, some have 1, etc.).

Solution: use a regex-based match in the join condition, stripping leading zeros via regexp_replace:

python
from pyspark.sql.functions import regexp_replace, col

df1_clean = table1.withColumn("id_clean", regexp_replace(col("id"), "^0+", ""))
df2_clean = table2.withColumn("id_clean", regexp_replace(col("id"), "^0+", ""))

result = df1_clean.join(df2_clean, "id_clean")

regexp_replace(col, "^0+", "") removes any number of leading zeros, regardless of how many there are — "0001" → "1", "001" → "1", "1" stays "1" (no leading zeros to strip). This technically isn't "casting" or "substring," satisfying the constraint, while correctly normalizing any inconsistent padding.

SQL equivalent:

sql
SELECT * FROM table1 t1
JOIN table2 t2
ON REGEXP_REPLACE(t1.id, '^0+', '') = REGEXP_REPLACE(t2.id, '^0+', '')